# Hybrid Models in Series

In this example, we'll explore **serial hybrid modeling**, where we combine a mechanistic model with a machine learning model to correct for missing mechanisms.

## The Idea

Sometimes our mechanistic models are incomplete - we may know some of the mechanisms, but not all of it. In such cases, we can:

1. Build a mechanistic model with the physics we *do* know
2. Use it to make predictions and observe systematic errors (residuals)
3. Train a machine learning model to predict those residuals from features
4. Combine both: **y_hybrid = y_mechanistic + y_ML_correction**

This approach lets us leverage both physical knowledge and data-driven learning.

## Our Example: The Missing Inhibitor

- **True system**: Has inhibitor binding (I + E ⇌ EI) that slows down the reaction
- **Mechanistic model**: Ignores the inhibitor (simpler, incomplete model)
- **ML correction**: Learn residuals as a function of inhibitor concentration and system state
- **Test generalization**: Train on some inhibitor concentrations, test on others

In [ ]:
using OrdinaryDiffEq, CairoMakie
using Random
using Statistics: mean
using DataFrames
using XGBoost

## The "True" System: Enzyme Kinetics with Inhibitor

Recall from Tutorial 1 the basic enzyme-substrate reaction:

$$ S + E \xrightleftharpoons[k_{-1}]{k_1} ES \xrightarrow{k_2} P + E $$

In the DIY exercise, we added an inhibitor that binds to the enzyme:

$$ I + E \xrightleftharpoons[k_{-3}]{k_3} EI $$

The complete system has 6 species and these ODEs:

\begin{align*}
\frac{d[S]}{dt} &= -k_1 [S][E] + k_{-1} [ES] \\
\frac{d[E]}{dt} &= -k_1 [S][E] + (k_{-1} + k_2) [ES] - k_3 [I][E] + k_{-3} [EI] \\
\frac{d[ES]}{dt} &= k_1 [S][E] - (k_{-1} + k_2) [ES] \\
\frac{d[P]}{dt} &= k_2 [ES] \\
\frac{d[I]}{dt} &= -k_3 [I][E] + k_{-3} [EI] \\
\frac{d[EI]}{dt} &= k_3 [I][E] - k_{-3} [EI]
\end{align*}

This extended model will be our **ground truth** - it represents the real system.

In [ ]:
function enzyme_with_inhibitor!(du, u, p, t)
    k1, kn1, k2, k3, kn3 = p
    S, E, ES, P, I, EI = u

    du[1] = -k1 * S * E + kn1 * ES                           # dS/dt
    du[2] = -k1 * S * E + (kn1 + k2) * ES - k3 * I * E + kn3 * EI  # dE/dt
    du[3] = k1 * S * E - (kn1 + k2) * ES                     # dES/dt
    du[4] = k2 * ES                                          # dP/dt
    du[5] = -k3 * I * E + kn3 * EI                          # dI/dt
    du[6] = k3 * I * E - kn3 * EI                           # dEI/dt
end

And the simpler mechanistic model that **ignores the inhibitor** (this will be our incomplete mechanistic model):

In [ ]:
function enzyme_simple!(du, u, p, t)
    k1, kn1, k2 = p
    S, E, ES, P = u

    du[1] = -k1 * S * E + kn1 * ES
    du[2] = -k1 * S * E + (kn1 + k2) * ES
    du[3] = k1 * S * E - (kn1 + k2) * ES
    du[4] = k2 * ES
end

## Generating Training Data: Multiple Trajectories

For machine learning to work, we need **multiple trajectories** exploring different conditions. We'll vary the initial inhibitor concentration I₀ to create different scenarios.

Key insight: I₀ is a known **experimental condition** (we know how much inhibitor we added), even though our mechanistic model doesn't include inhibitor mechanism.

In [ ]:
# Set random seed for reproducibility
Random.seed!(1234)

# True parameters (including inhibitor kinetics)
p_true = [1.1, 0.5, 0.9, 0.8, 0.3]  # [k1, kn1, k2, k3, kn3]

# Fixed mechanistic parameters (no inhibitor knowledge)
p_mech = [1.1, 0.5, 0.9]  # [k1, kn1, k2]

# Initial conditions and time span
S0 = 1.0
E0 = 0.8
tspan = (0.0, 20.0)
t_sample = 0.0:0.5:20.0

# Training inhibitor concentrations
I0_train = [1.0, 2.0, 3.0, 4.0]

# Validation inhibitor concentrations (for hyperparameter tuning)
I0_val = [1.1, 1.5, 3.5]

# Test inhibitor concentrations (final evaluation only)
# - 2.0: interpolation (within training range 0-4)
# - 6.0: extrapolation (beyond training range)
I0_test = [2.5, 6.0]

println("Training on I₀:   ", I0_train)
println("Validation on I₀: ", I0_val)
println("Testing on I₀:    ", I0_test, " (interpolation: 2.0, extrapolation: 6.0)")

Generate the "true" data using the full model with inhibitor:

In [ ]:
# Generate the 'measured' trajectories / datapoints with noise
true_solutions = []
noise_level = 0.02

for I0 in vcat(I0_train, I0_val, I0_test)
    u0_true = [S0, E0, 0.0, 0.0, I0, 0.0]  # [S, E, ES, P, I, EI]
    
    prob_true = ODEProblem(enzyme_with_inhibitor!, u0_true, tspan, p_true)
    sol_true = solve(prob_true, Tsit5(), saveat=t_sample)
    
    # Add measurement noise to S and P
    S_noisy = max.(0.0, sol_true[1, :] .+ randn(length(t_sample)) .* noise_level)
    P_noisy = max.(0.0, sol_true[4, :] .+ randn(length(t_sample)) .* noise_level)
    
    push!(true_solutions, (I0=I0, sol=sol_true, S_measured=S_noisy, P_measured=P_noisy))
end

println("✓ Generated $(length(true_solutions)) trajectories")

Let's visualize the effect of inhibitor on product formation:

In [ ]:
fig_true = let f = Figure(size=(900, 400))
    ax1 = Axis(f[1,1], 
        title="True System: Effect of Inhibitor on Product Formation",
        xlabel="Time (min)", 
        ylabel="Product Concentration [P]",
        titlealign=:left)
    
    # Track which labels we've added
    added_train = false
    added_val = false
    added_test = false
    
    # Plot training data
    for data in true_solutions
        if data.I0 in I0_train
            lines!(ax1, data.sol.t, data.sol[4, :], 
                   color=:blue, linewidth=2, alpha=0.5)
            scatter!(ax1, t_sample, data.P_measured, 
                    color=:blue, markersize=4, alpha=0.6, 
                    label=added_train ? nothing : "Train")
            added_train = true
        end
    end
    
    # Plot validation data
    for data in true_solutions
        if data.I0 in I0_val
            lines!(ax1, data.sol.t, data.sol[4, :], 
                   color=:orange, linewidth=2, alpha=0.5)
            scatter!(ax1, t_sample, data.P_measured, 
                    color=:orange, markersize=4, alpha=0.6,
                    label=added_val ? nothing : "Val")
            added_val = true
        end
    end
    
    # Plot test data
    for data in true_solutions
        if data.I0 in I0_test
            lines!(ax1, data.sol.t, data.sol[4, :], 
                   color=:red, linewidth=2, alpha=0.5)
            scatter!(ax1, t_sample, data.P_measured, 
                    color=:red, markersize=4, alpha=0.6,
                    label=added_test ? nothing : "Test")
            added_test = true
        end
    end
    
    axislegend(ax1, position=:rb)
    f
end

Same data, colored by inhibitor concentration instead:

In [ ]:
fig_true_by_I0 = let f = Figure(size=(900, 400))
    ax1 = Axis(f[1,1], 
        title="True System: Effect of Inhibitor on Product Formation",
        xlabel="Time (min)", 
        ylabel="Product Concentration [P]",
        titlealign=:left)
    
    # Get all I0 values and create colormap
    all_I0 = sort(vcat(I0_train, I0_val, I0_test))
    colors = get(Makie.ColorSchemes.viridis, range(0.0, 1.0, length=length(all_I0)))
    
    # Plot all trajectories colored by I0
    for (i, I0) in enumerate(all_I0)
        data_idx = findfirst(d -> d.I0 == I0, true_solutions)
        if !isnothing(data_idx)
            data = true_solutions[data_idx]
            lines!(ax1, data.sol.t, data.sol[4, :], 
                   color=colors[i], linewidth=2, alpha=0.7)
            scatter!(ax1, t_sample, data.P_measured, 
                    color=colors[i], markersize=4, alpha=0.6)
        end
    end
    
    Colorbar(f[1,2], colormap=:viridis, limits=(minimum(all_I0), maximum(all_I0)), label="I₀")
    f
end

## Mechanistic Model Predictions

Now we use our **simple mechanistic model** (without inhibitor) to make predictions for all trajectories. This model doesn't know about the inhibitor, so it will show systematic errors.

In [ ]:
# Solve mechanistic model (once, with fixed parameters)
u0_mech = [S0, E0, 0.0, 0.0]  # [S, E, ES, P]
prob_mech = ODEProblem(enzyme_simple!, u0_mech, tspan, p_mech)
sol_mech = solve(prob_mech, Tsit5(), saveat=t_sample)

println("✓ Solved simple mechanistic model")

## Computing Residuals

The mechanistic model makes the same prediction regardless of I₀ (it doesn't know about the inhibitor). Let's compute the residuals to see the systematic errors (colored by inhibitor concentration):

In [ ]:
fig_residuals = let f = Figure(size=(900, 400))
    ax = Axis(f[1,1],
        title="Residuals: True Data - Mechanistic Prediction",
        xlabel="Time (min)",
        ylabel="Residual [P]",
        titlealign=:left)
    
    # Get all I0 values and create colormap
    all_I0 = sort(vcat(I0_train, I0_val, I0_test))
    colors = get(Makie.ColorSchemes.viridis, range(0.0, 1.0, length=length(all_I0)))
    
    # Plot residuals colored by I0
    for (i, I0) in enumerate(all_I0)
        data_idx = findfirst(d -> d.I0 == I0, true_solutions)
        if !isnothing(data_idx)
            data = true_solutions[data_idx]
            residuals = data.P_measured .- sol_mech[4, :]
            lines!(ax, t_sample, residuals,
                   color=colors[i], linewidth=2, alpha=0.7)
        end
    end
    
    hlines!(ax, [0.0], color=:black, linestyle=:dash, alpha=0.5)
    Colorbar(f[1,2], colormap=:viridis, limits=(minimum(all_I0), maximum(all_I0)), label="I₀")
    f
end

The residuals show clear **structure**: more negative for higher I₀. This is what we'll need to learn.
## Preparing Training Data for Machine Learning

Create a tabular dataset with:
- **Features**: time, observable mechanistic predictions (S, P), and I₀
- **Target**: residual = P_true - P_mechanistic

**Important**: We only use **observable quantities** as features:
- Time, substrate, and product are measurable
- Internal enzyme states (E, ES) are NOT directly observable in experiments

**Data split strategy**:
- **Train**: Learn the correction pattern
- **Validation**: Tune hyperparameters (used in XGBoost watchlist)
- **Test**: Final evaluation with two held-out trajectories:
  - I₀ = 2.0 (interpolation within training range)
  - I₀ = 6.0 (extrapolation beyond training range)

In [ ]:
# Collect data split by train/val/test
train_data = []
val_data = []
test_data = []

for data in true_solutions
    for (i, t) in enumerate(t_sample)
        # Features: only observable quantities
        S_mech = sol_mech[1, i]  # Observable: substrate
        P_mech = sol_mech[4, i]  # Observable: product
        # NOT using E_mech, ES_mech - these are internal states not typically measurable
        
        # True measurement
        P_true = data.P_measured[i]
        
        # Residual (target)
        residual = P_true - P_mech
        
        point = (
            t = t,
            S_mech = S_mech,
            P_mech = P_mech,
            I0 = data.I0,
            P_true = P_true,
            residual = residual
        )
        
        if data.I0 in I0_train
            push!(train_data, point)
        elseif data.I0 in I0_val
            push!(val_data, point)
        else
            push!(test_data, point)
        end
    end
end

df_train = DataFrame(train_data)
df_val = DataFrame(val_data)
df_test = DataFrame(test_data)

println("✓ Training dataset:   $(nrow(df_train)) samples from I₀ ∈ ", I0_train)
println("✓ Validation dataset: $(nrow(df_val)) samples from I₀ ∈ ", I0_val)
println("✓ Test dataset:       $(nrow(df_test)) samples from I₀ ∈ ", I0_test)

In [ ]:
# Only use observable features (not internal enzyme states)
feature_names = [:t, :S_mech, :P_mech, :I0]

X_train = Matrix(select(df_train, feature_names))
y_train = df_train.residual

X_val = Matrix(select(df_val, feature_names))
y_val = df_val.residual

X_test = Matrix(select(df_test, feature_names))
y_test = df_test.residual

println("Features: ", feature_names)

## Train XGBoost Model

We'll train on data from **training inhibitor concentrations** only:

In [ ]:
using Pkg
Pkg.status()

In [ ]:
dtrain = DMatrix(X_train, label=y_train)
dval = DMatrix(X_val, label=y_val)

bst = xgboost(
    dtrain, 300;
    #num_round=300,
    #watchlist=Dict("train" => dtrain, "val" => dval),
    objective="reg:squarederror",
    eval_metric="rmse",
    max_depth=2,
    eta=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

println("✓ Model trained!")

In [ ]:
# Evaluate on test set (two specific trajectories)
dtest = DMatrix(X_test)
residual_pred = predict(bst, dtest)

# Hybrid predictions
P_mech_test = df_test.P_mech
P_true_test = df_test.P_true

P_hybrid = P_mech_test .+ residual_pred

# Metrics
rmse_mech = sqrt(mean((P_true_test .- P_mech_test).^2))
rmse_hybrid = sqrt(mean((P_true_test .- P_hybrid).^2))

println("\n" * "="^60)
println("Performance on HELD-OUT test trajectories:")
println("="^60)
println("Test I₀ = 2.5 (interpolation, within training range)")
println("Test I₀ = 6.0 (extrapolation, beyond training range)")
println("-"^60)
println("Mechanistic RMSE = $(round(rmse_mech, digits=5))")
println("Hybrid RMSE      = $(round(rmse_hybrid, digits=5))")
println("Improvement      = $(round((1 - rmse_hybrid/rmse_mech)*100, digits=1))%")

## Visualize Hybrid Model Performance

Let's compare predictions for both training and test inhibitor concentrations:

In [ ]:
fig_final = let f = Figure(size=(1000, 600))
    
    # Show all trajectories organized by train/val/test
    all_I0 = sort(vcat(I0_train, I0_val, I0_test))
    
    for (idx, I0) in enumerate(all_I0)
        row = div(idx - 1, 3) + 1
        col = mod(idx - 1, 3) + 1
        
        # Determine split
        if I0 in I0_train
            title_suffix = " (train)"
        elseif I0 in I0_val
            title_suffix = " (val)"
        else
            # Annotate interpolation vs extrapolation
            if I0 == 2.5
                title_suffix = " (test: interp)"
            else
                title_suffix = " (test: extrap)"
            end
        end
        
        ax = Axis(f[row, col],
            title="I₀ = $(I0)$title_suffix",
            xlabel="Time (min)",
            ylabel="[P]",
            titlealign=:left)
        
        # Find the true data
        data_idx = findfirst(d -> d.I0 == I0, true_solutions)
        if !isnothing(data_idx)
            data = true_solutions[data_idx]
            
            # Prepare features (only observables!)
            X_traj = hcat(
                collect(t_sample),
                sol_mech[1,:],  # S_mech
                sol_mech[4,:],  # P_mech
                fill(I0, length(t_sample))  # I0
            )
            
            dtraj = DMatrix(X_traj)
            residual_pred_traj = predict(bst, dtraj)
            P_hybrid_traj = sol_mech[4,:] .+ residual_pred_traj
            
            # Plot true data
            lines!(ax, t_sample, data.P_measured,
                   label="True", linewidth=2.5, color=:black)
            
            # Plot mechanistic model
            lines!(ax, t_sample, sol_mech[4,:],
                   label="Mechanistic", linewidth=2, linestyle=:dash,
                   color=Makie.wong_colors()[2])
            
            # Plot hybrid model
            lines!(ax, t_sample, P_hybrid_traj,
                   label="Hybrid", linewidth=2,
                   color=Makie.wong_colors()[3])
            
            if idx == 1
                axislegend(ax, position=:rb)
            end
        end
    end
    
    f
end

## Visualizing Learned Correction

Let's see how the XGBoost model's learned correction varies with inhibitor concentration (take ~15 values I0 ∈ [0.0, 7.0]):

In [ ]:
fig_learned = let f = Figure(size=(900, 400))
    ax = Axis(f[1,1],
        title="XGBoost Learned Correction vs Inhibitor Concentration",
        xlabel="Time (min)",
        ylabel="Predicted Residual",
        titlealign=:left)
    
    # Create a range of I0 values to visualize
    I0_range = 0.0:0.5:7.0
    
    colors = get(Makie.ColorSchemes.viridis, range(0.0, 1.0, length=length(I0_range)))
    
    for (i, I0) in enumerate(I0_range)
        # Prepare features for this I0 trajectory
        X_pred = hcat(
            collect(t_sample),
            sol_mech[1,:],  # S_mech
            sol_mech[4,:],  # P_mech
            fill(I0, length(t_sample))
        )
        
        dpred = DMatrix(X_pred)
        residual_pred = predict(bst, dpred)
        
        lines!(ax, t_sample, residual_pred,
               color=colors[i], linewidth=2,
               label="I₀ = $(I0)")
    end
    
    hlines!(ax, [0.0], color=:black, linestyle=:dash, alpha=0.5)
    
    Colorbar(f[1,2], colormap=:viridis, limits=(0.0, 7.0), label="I₀")
    
    f
end

**Observation**: Notice that XGBoost doesn't learn a smooth continuum across I₀ values. Instead, the predicted corrections show **step-like behavior** - the model essentially picks the "closest" training trajectory rather than smoothly interpolating.

This is a fundamental limitation of tree-based models like XGBoost:
- Trees split on feature values, creating piecewise constant predictions
- Since I₀ only appears at discrete training values (1, 2, 3, 4), the model learns discrete bins
- Interpolation happens via averaging similar training samples, not smooth functions

For problems requiring smooth interpolation with respect to experimental conditions, **neural networks** would be more appropriate (as we'll see in the UDE tutorial). However, XGBoost can still provide good predictions if:
- Test conditions are close to training conditions
- The relationship is relatively simple
- You have enough training diversity